# SVM Taxi Demand Prediction — Pooled Model (Community Area)

**One pooled SVM model per resolution, instead of one model per spatial unit.**

`03.04_prediction_svm_community_area.ipynb` fits a separate SVR per included community area
(21 of 77). That approach produced valuable findings (per-area heterogeneity, the value of a
second time-harmonic, a kernel comparison) and **stays in place as a backup / deep-dive** — this
notebook does not replace it and nothing there has changed.

This notebook takes a different architecture: **one model trained on all (area, time-window)
rows at once**, with spatial-unit identity supplied as an input (community-area fixed effects)
rather than fitting hundreds of separate models. Two reasons drove this:

1. **Scalability to census tracts.** Chicago has ~800 census tracts. Fitting a full
   kernel-ladder + GridSearchCV pipeline per unit, as `03.04` does for 77 areas, does not scale
   to that many units, nor does it scale to whatever finer resolution a follow-up might use.
   A pooled model scales by re-aggregating the panel at a different spatial granularity and
   refitting once, not by refitting hundreds of times.
2. **Comparability.** The task requires (a) comparing performance across spatial resolutions
   (community area vs. census tract) and (b) comparing SVM against a feedforward neural
   network on the same resolution. A feedforward NN is naturally a single pooled model over
   engineered features — it is not trained as hundreds of separate per-unit networks. Using a
   pooled architecture for SVM now means the same panel-building, train/test split, and
   skill-score evaluation code can be reused directly for both the census-tract comparison and
   the NN notebook, so the final comparisons are genuinely apples-to-apples.

**Scope of this first draft:** community-area resolution only, to validate the pooled
architecture before extending to census tracts (outlined, not yet implemented, at the end).

**Course constraint (same as `03.04`):** no lagged / autoregressive demand features — only
exogenous time, weather, and spatial-identity inputs.


## Validation Strategy

Identical to `03.04`, reused here for direct comparability:

- **Chronological holdout**: train 2024-01-01 - 2025-12-31 (two full years), test 2026-01-01 -
  2026-05-31 (five months) — the model is fit on the past, asked about the future.
- **`TimeSeriesSplit` (3 folds)** for all cross-validation / GridSearch, so hyperparameter
  selection never look ahead.
- **Complete area x window grid**, zero-filled, so both training and evaluation cover genuine
  "zero demand" windows.
- **Profile baseline + skill score** (`1 - MAE_model / MAE_baseline`) per area, exactly as in
  `03.04` — the headline metric, since it is comparable across areas of very different demand
  scale (and, later, across resolutions).

**Difference from `03.04`:** no demand threshold / area exclusion here. Pooling is exactly the
mechanism that lets low-volume areas participate (they contribute rows and share the global
model's learned relationships, plus their own fixed effect) without needing a separate model
fit just for them.


In [ ]:
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

from sklearn.svm import SVR, LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TAXI_DATA_PATH       = "../data/chicago_taxi_2024_2026_clean.parquet"
WEATHER_DATA_PATH    = "../data/chicago_weather_2024_2026_hourly.csv"
COMMUNITY_AREAS_PATH = "../data/community_areas_chicago.geojson"

# Data range covered by the cleaned parquet (June 2026+ excluded: portal reporting lag)
DATA_START    = pd.Timestamp("2024-01-01")
DATA_END_EXCL = pd.Timestamp("2026-06-01")

# Chronological split: train 2024-2025 (two full years), test Jan-May 2026
TRAIN_END = pd.Timestamp("2026-01-01")

FREQ = "4h"  # this draft: community-area resolution at 4h windows only
WEATHER_COLS = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "cloud_cover"]

ALL_AREAS = np.arange(1, 78)  # official community areas 1..77 -- ALL of them, no threshold

# expm1(709) overflows float64; clip the log-scale prediction first so an unstable kernel
# (see the kernel comparison below) gets a very bad but finite score instead of crashing.
LOG_PRED_CAP = 20


## 1. Data Loading

Identical source and cleaning logic to `03.04` — see `00_data_loading` for how the combined
2024-2026 parquet was produced. Only the pickup community area and trip timestamp are needed
for demand aggregation.


In [ ]:
taxi = pd.read_parquet(TAXI_DATA_PATH, columns=["trip_start_timestamp", "pickup_community_area"])
n_raw = len(taxi)
taxi = taxi[taxi["pickup_community_area"].notna()].copy()
taxi["pickup_community_area"] = taxi["pickup_community_area"].astype(int)
print(f"Trips with pickup CA: {len(taxi):,} of {n_raw:,} ({len(taxi)/n_raw*100:.1f}%)")

weather = pd.read_csv(WEATHER_DATA_PATH)
weather["datetime"] = pd.to_datetime(weather["datetime"])
weather = weather[["datetime"] + WEATHER_COLS]
print(f"Weather rows: {len(weather):,}")


## 2. Building the Pooled Demand Panel

Same complete zero-filled area x window grid as `03.04` section 2 — **but with every one of
the 77 areas retained** (no demand-threshold exclusion; pooling is how low-volume areas get a
usable prediction here).

Time features include the **second harmonic** for hour and day-of-week
(`sin/cos(4*pi*t/period)` alongside the usual `sin/cos(2*pi*t/period)`) from the start — `03.04`
section 7.3 validated that this closes a real, measurable part of the gap to the profile
baseline (a single harmonic cannot represent a bimodal morning+evening rush pattern).

Spatial identity is encoded as **community-area fixed effects**: one-hot dummies, matching the
"Variant B" global model already tested in `03.04` section 6, which raised median per-area R2
from 0.005 (POI counts alone) to 0.191 (POI + area dummies) versus the profile baseline. Since
every census tract nests inside exactly one of the 77 community areas, these same dummy columns
carry forward directly when this notebook is extended to tract level.


In [ ]:
US_HOLIDAYS = pd.to_datetime([
    "2024-01-01", "2024-01-15", "2024-02-19", "2024-05-27",
    "2024-06-19", "2024-07-04", "2024-09-02", "2024-10-14",
    "2024-11-11", "2024-11-28", "2024-12-25",
    "2025-01-01", "2025-01-20", "2025-02-17", "2025-05-26",
    "2025-06-19", "2025-07-04", "2025-09-01", "2025-10-13",
    "2025-11-11", "2025-11-27", "2025-12-25",
    "2026-01-01", "2026-01-19", "2026-02-16", "2026-05-25",
])

window = taxi["trip_start_timestamp"].dt.floor(FREQ).rename("timestamp")
counts = taxi.groupby(["pickup_community_area", window]).size().rename("count")

all_windows = pd.date_range(DATA_START, DATA_END_EXCL, freq=FREQ, inclusive="left")
grid = pd.MultiIndex.from_product([ALL_AREAS, all_windows], names=["community_area", "timestamp"])
panel = counts.reindex(grid, fill_value=0).reset_index()

w = weather.copy()
w["timestamp"] = w["datetime"].dt.floor(FREQ)
w_agg = w.groupby("timestamp")[WEATHER_COLS].mean().reset_index()
panel = panel.merge(w_agg, on="timestamp", how="left")

panel["hour"]        = panel["timestamp"].dt.hour
panel["day_of_week"] = panel["timestamp"].dt.dayofweek
month                = panel["timestamp"].dt.month

panel["hour_sin"]         = np.sin(2 * np.pi * panel["hour"] / 24)
panel["hour_cos"]         = np.cos(2 * np.pi * panel["hour"] / 24)
panel["hour_sin2"]        = np.sin(4 * np.pi * panel["hour"] / 24)   # 2nd harmonic (03.04 7.3)
panel["hour_cos2"]        = np.cos(4 * np.pi * panel["hour"] / 24)
panel["day_of_week_sin"]  = np.sin(2 * np.pi * panel["day_of_week"] / 7)
panel["day_of_week_cos"]  = np.cos(2 * np.pi * panel["day_of_week"] / 7)
panel["day_of_week_sin2"] = np.sin(4 * np.pi * panel["day_of_week"] / 7)
panel["day_of_week_cos2"] = np.cos(4 * np.pi * panel["day_of_week"] / 7)
panel["month_sin"]        = np.sin(2 * np.pi * month / 12)
panel["month_cos"]        = np.cos(2 * np.pi * month / 12)
panel["is_weekend"]       = (panel["day_of_week"] >= 5).astype(int)
panel["is_holiday"]       = panel["timestamp"].dt.date.isin(US_HOLIDAYS.date).astype(int)

# Community-area fixed effects (spatial identity)
dummies = pd.get_dummies(panel["community_area"], prefix="area", drop_first=True).astype(float)
DUMMY_COLS = list(dummies.columns)
panel = pd.concat([panel, dummies], axis=1)

panel = panel.sort_values(["community_area", "timestamp"]).reset_index(drop=True)
print(f"{FREQ}: {len(panel):,} rows ({panel['community_area'].nunique()} areas x "
      f"{panel['timestamp'].nunique():,} windows), {len(DUMMY_COLS)} area dummies, "
      f"zero-demand windows: {(panel['count']==0).mean()*100:.1f}%")


## 3. Feature Set & Shared Helpers

17 exogenous time/weather features (13 from `03.04` + the 4 second-harmonic terms) plus 76
community-area dummies.

**A scaling detail that matters more than it looks:** `StandardScaler` must **not** be applied
to the 0/1 dummy columns. Standardizing a rare binary column inflates its scale (small std ->
large `1/std`), which (a) made `LinearSVR`'s solver converge far slower in testing (~1076s vs.
~205s for the identical fit, just with dummies left unscaled), and (b) distorted the RBF
kernel's distance calculation enough to change which kernel looked best in the comparison below.
A `ColumnTransformer` scales only the continuous features and passes the dummies through as-is.


In [ ]:
FEATURE_COLS = [
    "temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "cloud_cover",
    "hour_sin", "hour_cos", "hour_sin2", "hour_cos2",
    "day_of_week_sin", "day_of_week_cos", "day_of_week_sin2", "day_of_week_cos2",
    "month_sin", "month_cos", "is_weekend", "is_holiday",
]
ALL_COLS = FEATURE_COLS + DUMMY_COLS

preprocess = ColumnTransformer([
    ("scale", StandardScaler(), FEATURE_COLS),
    ("area_dummies", "passthrough", DUMMY_COLS),
])

train = panel[panel["timestamp"] < TRAIN_END]
test  = panel[panel["timestamp"] >= TRAIN_END]
y_true = test["count"].to_numpy(dtype=float)
print(f"train={len(train):,} rows   test={len(test):,} rows")


def metrics(y_true, y_pred):
    return {
        "r2":   round(r2_score(y_true, y_pred), 4),
        "mae":  round(mean_absolute_error(y_true, y_pred), 2),
        "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 2),
    }


def predict_counts(model, X):
    """Back-transform log-scale predictions to counts, clipped at 0 (and capped pre-expm1;
    see 03.04 section 4 for why an unclipped kernel prediction can overflow to inf)."""
    y_log = np.clip(model.predict(X), -LOG_PRED_CAP, LOG_PRED_CAP)
    return np.clip(np.expm1(y_log), 0, None)


def per_area_results(test_df, y_pred):
    """One row per community area: MAE/R2/RMSE of the pooled model's predictions for that area."""
    tmp = pd.DataFrame({
        "community_area": test_df["community_area"].to_numpy(),
        "y_true": test_df["count"].to_numpy(dtype=float),
        "y_pred": y_pred,
    })
    rows = [
        {"community_area": int(area), **metrics(g["y_true"], g["y_pred"])}
        for area, g in tmp.groupby("community_area")
    ]
    return pd.DataFrame(rows)


def skill_vs_baseline(results_df):
    """Per-area skill score: 1 - MAE_model / MAE_baseline (see 03.04 section 4 for the full
    rationale -- comparable across areas of different scale; MAE rather than RMSE so the
    baseline, which predicts the group mean, gets no built-in advantage under the scoring loss)."""
    base = baseline_results[["community_area", "mae"]].rename(columns={"mae": "mae_base"})
    merged = results_df.merge(base, on="community_area")
    return (1 - merged["mae"] / merged["mae_base"]).rename("skill")


AREA_MEDIAN_DEMAND = panel.groupby("community_area")["count"].median().rename("median_demand")


def weighted_skill_vs_baseline(results_df):
    """Demand-weighted mean skill: each area's skill weighted by its own median demand, so
    high-volume areas -- where the fleet is actually deployed -- drive the number. This is the
    headline metric here (business case: the fleet is allocated mostly to high-demand areas).
    The unweighted median (every area counted equally) is kept as a secondary spatial-coverage
    view -- see 03.04 section 14 for a worked example of how much these two views can diverge.
    """
    skill = skill_vs_baseline(results_df)
    weights = results_df[["community_area"]].merge(
        AREA_MEDIAN_DEMAND, left_on="community_area", right_index=True
    )["median_demand"].to_numpy()
    return float(np.average(skill.to_numpy(), weights=weights))


def print_pooled_results(label, pooled_metrics, per_area_df):
    w_skill = weighted_skill_vs_baseline(per_area_df)
    print(label)
    print(f"  Pooled  : R2={pooled_metrics['r2']:.3f}  MAE={pooled_metrics['mae']:.2f}  "
          f"RMSE={pooled_metrics['rmse']:.2f}")
    print(f"  Per area: demand-weighted skill (headline)={w_skill:+.3f}  "
          f"unweighted median skill={per_area_df['skill'].median():+.3f}  "
          f"median R2={per_area_df['r2'].median():.3f}  "
          f"(areas with negative skill: {(per_area_df['skill']<0).sum()} of {len(per_area_df)})")


## 4. Profile Baseline

Same construction as `03.04` section 5: for each area, the train-set mean demand for that
(hour, weekday) combination. Computed for all 77 areas (no exclusion), since every area needs
its own skill-score reference now.


In [ ]:
def profile_baseline(panel, areas):
    results = []
    for area in areas:
        df_area = panel[panel["community_area"] == area]
        tr = df_area[df_area["timestamp"] < TRAIN_END]
        te = df_area[df_area["timestamp"] >= TRAIN_END]
        profile = tr.groupby(["hour", "day_of_week"])["count"].mean()
        fallback = tr["count"].mean()
        keys = pd.MultiIndex.from_arrays([te["hour"], te["day_of_week"]])
        y_pred = profile.reindex(keys).fillna(fallback).to_numpy()
        y_true_ = te["count"].to_numpy(dtype=float)
        results.append({
            "community_area": int(area),
            "median_demand": float(df_area["count"].median()),
            **metrics(y_true_, y_pred),
        })
    return pd.DataFrame(results)


t0 = time.time()
baseline_results = profile_baseline(panel, ALL_AREAS)
print(f"Profile baseline, all 77 areas ({time.time()-t0:.0f}s): "
      f"median R2={baseline_results['r2'].median():.3f}  median MAE={baseline_results['mae'].median():.2f}")


## 5. Pooled Model Without a Kernel (LinearSVR) — the Assignment's Starting Point

*"Simply start without a kernel."* One `LinearSVR`, fit once on **all 337,722 training rows**
(all 77 areas, both years), evaluated pooled and per area.


In [ ]:
t0 = time.time()
linear_model = Pipeline([
    ("prep", preprocess),
    ("svm", LinearSVR(C=1.0, epsilon=0.1, max_iter=20000, random_state=42)),
])
linear_model.fit(train[ALL_COLS], np.log1p(train["count"]))
y_pred_linear = predict_counts(linear_model, test[ALL_COLS])
pm_linear = metrics(y_true, y_pred_linear)
res_linear = per_area_results(test, y_pred_linear)
res_linear["skill"] = skill_vs_baseline(res_linear)

print_pooled_results(f"Pooled LinearSVR ({time.time()-t0:.0f}s)", pm_linear, res_linear)


## 6. Kernel Comparison

*"Then, gradually make your model complex by integrating different kinds of kernels."*

Exact (non-approximate) kernel SVR is O(n^2)-O(n^3) in training rows. On 337,722 pooled rows
that is not feasible (an early attempt at a scalable approximation -- kernel-approximation
features via `Nystroem` + `LinearSVR` on the full data -- produced numerically unstable,
nonsensical predictions and is not included here; it is flagged as unresolved future work in
section 9). Kernels are therefore compared on a **6,000-row random subsample** of the training
data, still evaluated against the **full, untouched test set**.

**Read this comparison as informational only, not as a selection procedure** — the next section
shows that properly cross-validating on a subsample can pick a very different, and much worse,
hyperparameter combination than the one that happens to look best on a single subsample draw.


In [ ]:
rng = np.random.RandomState(42)
sub_idx = rng.choice(train.index, size=6000, replace=False)
train_sub = train.loc[sub_idx]

candidates = {
    "no kernel (LinearSVR)": LinearSVR(C=1.0, epsilon=0.1, max_iter=20000, random_state=42),
    "linear (SVR)":          SVR(kernel="linear",  C=100, epsilon=0.1),
    "poly, degree 2":        SVR(kernel="poly",    degree=2, C=100, gamma="scale", epsilon=0.1),
    "rbf":                   SVR(kernel="rbf",     C=100, gamma="scale", epsilon=0.1),
    "sigmoid":               SVR(kernel="sigmoid", C=100, gamma="scale", epsilon=0.1),
}

kernel_rows = []
for name, svm in candidates.items():
    t0 = time.time()
    m = Pipeline([("prep", preprocess), ("svm", svm)])
    m.fit(train_sub[ALL_COLS], np.log1p(train_sub["count"]))
    y_pred = predict_counts(m, test[ALL_COLS])
    kernel_rows.append({"kernel": name, **metrics(y_true, y_pred), "fit_seconds": round(time.time() - t0, 1)})

kernel_df = pd.DataFrame(kernel_rows).sort_values("r2", ascending=False)
print("Kernel comparison -- 6,000-row train subsample, full test set, pooled metrics:")
print(kernel_df.to_string(index=False))


**A methodological trap, caught before finalizing this notebook:** the first version of this
comparison scaled the dummy columns along with the continuous features. Under that (incorrect)
preprocessing, RBF looked best (R2 = 0.850) and poly degree 2 looked mediocre (R2 = 0.363).
Once the dummies were correctly left unscaled, the ranking **flipped**: poly degree 2 became the
best kernel (R2 = 0.875) and RBF dropped to R2 = 0.738. The reason: RBF's kernel is a function of
raw Euclidean distance, which the inflated dummy scale was dominating (accidentally producing a
crude "cluster by area" effect); poly's kernel is a function of dot products and was far less
sensitive to this. **Lesson: how one-hot spatial identity columns are scaled can change which
kernel looks best, not just how good the winning one looks.** Sigmoid is unstable regardless of
scaling, consistent with `03.04`.


## 7. Hyperparameter Tuning — GridSearchCV with TimeSeriesSplit

The naive comparison above suggests poly degree 2. Tuning it **properly** — `GridSearchCV` with
`TimeSeriesSplit(3)`, on the same subsampling budget (10,000 rows, a different draw than
section 6) — tells a more honest and more important story than "poly wins."


In [ ]:
tscv = TimeSeriesSplit(n_splits=3)
rng2 = np.random.RandomState(42)
sub_idx2 = np.sort(rng2.choice(train.index, size=10000, replace=False))  # sorted: keep time order for TimeSeriesSplit
train_sub2 = train.loc[sub_idx2]

poly_param_grid = {"svm__C": [1, 10, 100], "svm__degree": [2, 3], "svm__gamma": ["scale"]}
t0 = time.time()
gs_poly = GridSearchCV(
    Pipeline([("prep", preprocess), ("svm", SVR(kernel="poly", epsilon=0.1))]),
    poly_param_grid, cv=tscv, scoring="r2", n_jobs=-1,
)
gs_poly.fit(train_sub2[FEATURE_COLS + DUMMY_COLS], np.log1p(train_sub2["count"]))
y_pred_poly = predict_counts(gs_poly, test[ALL_COLS])
pm_poly = metrics(y_true, y_pred_poly)
res_poly = per_area_results(test, y_pred_poly)
res_poly["skill"] = skill_vs_baseline(res_poly)

print(f"Tuned poly ({time.time()-t0:.0f}s): best_params={gs_poly.best_params_}  "
      f"best_cv_r2={gs_poly.best_score_:.3f}")
print_pooled_results("", pm_poly, res_poly)

rbf_param_grid = {"svm__C": [10, 100], "svm__gamma": ["scale", 0.01]}
t0 = time.time()
gs_rbf = GridSearchCV(
    Pipeline([("prep", preprocess), ("svm", SVR(kernel="rbf", epsilon=0.1))]),
    rbf_param_grid, cv=tscv, scoring="r2", n_jobs=-1,
)
gs_rbf.fit(train_sub2[FEATURE_COLS + DUMMY_COLS], np.log1p(train_sub2["count"]))
y_pred_rbf = predict_counts(gs_rbf, test[ALL_COLS])
pm_rbf = metrics(y_true, y_pred_rbf)
res_rbf = per_area_results(test, y_pred_rbf)
res_rbf["skill"] = skill_vs_baseline(res_rbf)

print(f"\nTuned RBF ({time.time()-t0:.0f}s): best_params={gs_rbf.best_params_}  "
      f"best_cv_r2={gs_rbf.best_score_:.3f}")
print_pooled_results("", pm_rbf, res_rbf)


**What happened, in the validation run behind this draft:** `GridSearchCV` picked
`{C: 1, degree: 2, gamma: scale}` for poly (`best_cv_r2 = 0.088` -- already mediocre in CV), and
that choice scored **R2 = -0.032 on the true test set** (median per-area skill -0.490, worse than
the profile baseline for 62 of 77 areas) -- despite `C=100, degree=2` looking excellent (R2=0.875)
on the single, uncross-validated subsample draw in section 6. `GridSearchCV` also tried
`C=100, degree=2` as one of its candidates and, within the 10,000-row subsample's `TimeSeriesSplit`
folds, judged it **worse** than `C=1`. Tuned RBF was more moderate: `best_cv_r2 = -0.079` (also
weak), test R2 = 0.792, median per-area skill -0.046 -- a small pooled-R2 improvement over
LinearSVR (0.778) but a slightly worse per-area skill.

**Conclusion for this draft: kernel-SVR hyperparameter selection from a small subsample of a
much larger pooled dataset is high-variance and not currently reliable here** -- a hyperparameter
combination's apparent quality swung from "best of five kernels" to "worse than the naive
baseline" depending on which subsample and validation procedure was used to judge it. The
`LinearSVR` fit on the full 337,722 rows (section 5) has no such instability (it is not
subsampled) and is the model actually carried into section 8's results. Closing this gap
properly needs either a scalable exact/approximate kernel method that can train on all the
data (the `Nystroem` attempt in section 9 was a first try and failed) or a much larger,
carefully validated subsample budget -- both are flagged as follow-up work, not resolved here.


## 8. Results — Pooled Model Comparison

`LinearSVR` (full data) is the model actually recommended by this draft, given section 7's
findings. The tuned kernel models are reported for completeness and transparency, not as a
replacement.


In [ ]:
comparison_df = pd.DataFrame([
    {"Model": "Profile baseline", "Demand-weighted skill": 0.000, "Median skill": 0.000,
     "Pooled R2": None, "Median R2/area": baseline_results["r2"].median()},
    {"Model": "LinearSVR (full data, no kernel)",
     "Demand-weighted skill": weighted_skill_vs_baseline(res_linear), "Median skill": res_linear["skill"].median(),
     "Pooled R2": pm_linear["r2"], "Median R2/area": res_linear["r2"].median()},
    {"Model": "RBF, tuned (10k subsample)",
     "Demand-weighted skill": weighted_skill_vs_baseline(res_rbf), "Median skill": res_rbf["skill"].median(),
     "Pooled R2": pm_rbf["r2"], "Median R2/area": res_rbf["r2"].median()},
    {"Model": "Poly, tuned (10k subsample)",
     "Demand-weighted skill": weighted_skill_vs_baseline(res_poly), "Median skill": res_poly["skill"].median(),
     "Pooled R2": pm_poly["r2"], "Median R2/area": res_poly["r2"].median()},
]).round(3)
print("Headline column is demand-weighted skill (the fleet is deployed mostly to high-demand")
print("areas, so those areas matter most); unweighted median skill and R2/MAE are supporting detail.")
print(comparison_df.to_string(index=False))

print("\nLinearSVR per-area results (sorted by skill vs baseline):")
print(res_linear.merge(AREA_MEDIAN_DEMAND, left_on="community_area", right_index=True)
      .sort_values("skill", ascending=False).to_string(index=False))


Per-area skill map for the recommended model (`LinearSVR`, full pooled data), same style as
`03.04` section 12 -- grey would indicate an excluded area, but nothing is excluded here.


In [ ]:
community_areas_gdf = gpd.read_file(COMMUNITY_AREAS_PATH)
community_areas_gdf["area_numbe"] = community_areas_gdf["area_numbe"].astype(int)

gdf_skill = community_areas_gdf.merge(
    res_linear[["community_area", "skill"]],
    left_on="area_numbe", right_on="community_area", how="left",
).to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(9, 9))
gdf_skill.plot(ax=ax, column="skill", cmap="RdYlGn", vmin=-0.5, vmax=0.5, legend=True,
               legend_kwds={"label": "Test skill vs. profile baseline (pooled LinearSVR, 4h)", "shrink": 0.6},
               edgecolor="white", linewidth=0.4)
ctx.add_basemap(ax, crs=gdf_skill.crs, source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.35)
ax.set_title("Pooled LinearSVR -- skill vs. baseline per community area (all 77 areas)")
ax.axis("off")
plt.tight_layout()
plt.show()


## 9. Shortfalls & Honest Limitations

- **Kernel-SVR tuning instability (section 7).** The single most important finding of this
  draft: a hyperparameter choice's apparent quality depended heavily on which small subsample
  and validation procedure was used to judge it. This is a real limitation of the
  subsample-then-tune workaround, not a property of kernel SVR in general -- with the full
  337,722 rows a kernel method might tune much more stably, but exact kernel SVR cannot
  train on that many rows in reasonable time.
- **The `Nystroem` kernel-approximation attempt failed.** To get a genuinely scalable
  (non-subsampled) non-linear model, `Nystroem(kernel="rbf", ...)` features feeding into
  `LinearSVR`, trained on the full pooled training data, were tried. The result was numerically
  broken (R2 in the negative hundreds of millions -- a handful of wildly extrapolated
  predictions dominating the metric, the same overflow pathology `03.04` section 4 documents
  for the sigmoid kernel). This needs a properly tuned `gamma`/`n_components` and likely a
  smaller `C` before it can be trusted, and is left as unresolved follow-up work rather than
  included half-working.
- **The dummy-scaling bug** (section 3/6) is a reminder that preprocessing choices for a mixed
  continuous + one-hot feature set are not cosmetic -- they changed both runtime (5x) and which
  kernel appeared best.
- **Fixed effects assume every unit is seen during training.** Community-area dummies work
  because the same 77 areas appear in both train and test. This assumption should still hold at
  census-tract level (same tracts, later time windows) but would break if the model needed to
  generalize to a genuinely unseen spatial unit -- not a concern for this task's validation
  strategy, but worth keeping in mind.
- **Rolling-origin backtest and 1-hour resolution** (present in `03.04`) are not yet repeated
  here -- natural next additions once the community-area draft is confirmed to be the right
  direction.


## 10. Next Steps — Census Tract Level (Outline, Not Yet Implemented)

Deliberately scoped out of this first draft per plan -- community area first, tract level once
this draft is validated. Outline for when that happens:

1. **Panel construction**: identical logic to section 2, joining trips to
   `census_tracts_chicago.geojson` (already in `data/`) instead of the community-area
   boundaries, producing a panel of (tract, window) rows instead of (area, window).
2. **Spatial identity, in two layers rather than ~800 flat dummies**: keep the 77 community-area
   dummies already built here as the coarse spatial signal (every tract nests inside exactly one
   community area), and add tract-level covariates (POI counts recomputed per tract, tract area,
   distance to downtown/O'Hare) for finer within-area distinction -- rather than one dummy per
   tract, which would not generalize and would be far more columns than needed.
3. **Row-count reality check**: ~800 tracts x 5,292 four-hour windows =~ 4.2M rows, ~10x this
   notebook's pooled panel. `LinearSVR` should still scale (it scaled fine here); the subsample
   sizes used for kernel comparison/tuning in sections 6-7 will need to grow accordingly, and the
   instability found in section 7 should be re-examined at tract scale before trusting any tuned
   kernel result there.
4. **Revisit the historical-aggregate-demand-as-feature question** (discussed but deliberately
   not used here) specifically if community-area dummies + tract covariates turn out not to be
   informative enough at tract level -- not before, since it would make the skill-vs-baseline
   comparison less meaningful (see discussion in conversation history / project memory).
5. Reuse this notebook's exact panel-building and evaluation code for the feedforward neural
   network notebook, swapping only the model-fitting step -- the whole point of the pooled
   architecture chosen here.
